In [0]:
path_1 = '/Volumes/it_support_tickets/source/input_files/it_support_tickets_typed_80_records.csv'

df = spark.read.csv(path_1, header=True, inferSchema=True)
display(df)

In [0]:
df.count()

In [0]:
df.printSchema()

In [0]:
df.columns
df.dtypes

In [0]:
from pyspark.sql.functions import current_timestamp, lit
bronze_df = df.withColumn('file_path', lit(path_1)) \
              .withColumn('ingestion_timestamp', current_timestamp()) \
              .withColumn('batch_id', lit(1))

In [0]:
display(bronze_df)

In [0]:
display(bronze_df.select("location").distinct())

In [0]:
#display invalid ticket_id
from pyspark.sql.functions import col
invalid_ticket_id = bronze_df.filter(
    col("ticket_id").isNull() |
    (~col("ticket_id").rlike("^TKT[0-9]{4}$"))
)

display(invalid_ticket_id)


In [0]:
# removing null values in date column
invalid_date = bronze_df.filter(
    col("created_date").isNull()
)

display(invalid_date)

In [0]:
# display invalid employee id
invalid_employee = bronze_df.filter(
    col("employee_id").isNull() |
    (~col("employee_id").rlike("^E[0-9]{3}$"))
)

display(invalid_employee)

In [0]:

invalid_category = bronze_df.filter(
    col("support_category").isNull()).show()

In [0]:
# displaying null values in priority column
invalid_priority = bronze_df.filter(
    col("priority").isNull() 
)

display(invalid_priority)

In [0]:
valid_engineers = [
    "Support Engineer 1",
    "Support Engineer 2",
    "Support Engineer 3",
    "Support Engineer 4",
    "Support Engineer 5",
    "Support Engineer 6"
]

invalid_engineer = bronze_df.filter(
    col("support_engineer").isNull() |
    (~col("support_engineer").isin(valid_engineers))
)

display(invalid_engineer)

In [0]:
valid_locations =[
    "Delhi",
    "Chennai",
    "Kochi",
    "Hyderabad",
    "Coimbatore",
    "Pune",
    "Mumbai",
    "Bengaluru"
]
invalid_location = bronze_df.filter(
    col("location").isNull() |
    (~col("location").isin(valid_locations))
)

display(invalid_location)

In [0]:
invalid_resolution = bronze_df.filter(
    col("resolution_hours").isNull() |
    (col("resolution_hours") <= 0)
)

display(invalid_resolution)

In [0]:
invalid_hourly_cost = bronze_df.filter(
    col("hourly_cost").isNull() |
    (col("hourly_cost") <= 0)
)

display(invalid_hourly_cost)

In [0]:
invalid_status = bronze_df.filter(
    col("ticket_status").isNull() |
    (col("ticket_status") == "INVALID_VALUE")
)

display(invalid_status)

In [0]:
print("Invalid Ticket IDs:", invalid_ticket_id.count())
print("Invalid Dates:", invalid_date.count())
print("Invalid Employees:", invalid_employee.count())
print("Invalid Priorities:", invalid_priority.count())
print("Invalid Engineers:", invalid_engineer.count())
print("Invalid Locations:", invalid_location.count())
print("Invalid Resolution Hours:", invalid_resolution.count())
print("Invalid Hourly Costs:", invalid_hourly_cost.count())
print("Invalid Status:", invalid_status.count())

In [0]:
# combining all invalid conditions from cells 9 to 19
from pyspark.sql.functions import col

valid_engineers = [
    "Support Engineer 1",
    "Support Engineer 2",
    "Support Engineer 3",
    "Support Engineer 4",
    "Support Engineer 5",
    "Support Engineer 6"
]

valid_locations = [
    "Delhi",
    "Chennai",
    "Kochi",
    "Hyderabad",
    "Coimbatore",
    "Pune",
    "Mumbai",
    "Bengaluru"
]

invalid_condition = (
    col("created_date").isNull() |
    col("employee_id").isNull() |
    (~col("employee_id").rlike("^E[0-9]{3}$")) |
    col("support_category").isNull() |
    col("priority").isNull() |
    col("support_engineer").isNull() |
    (~col("support_engineer").isin(valid_engineers)) |
    col("location").isNull() |
    (~col("location").isin(valid_locations)) |
    col("resolution_hours").isNull() |
    (col("resolution_hours") <= 0) |
    col("hourly_cost").isNull() |
    (col("hourly_cost") <= 0) |
    col("ticket_status").isNull() |
    (col("ticket_status") == "INVALID_VALUE")
)

display(invalid_condition)



In [0]:
bad_df = bronze_df.filter(invalid_condition)
display(bad_df)

In [0]:
good_df = bronze_df[~invalid_condition]
display(good_df)

In [0]:
# performing reconciliation
source_count = bronze_df.count()
good_df_count = good_df.count()
bad_df_count = bad_df.count()
print("Source Count: ", source_count)
print("Good Dataframe Count: ", good_df_count)
print("Bad Dataframe Count: ", bad_df_count)

if (source_count == (good_df_count + bad_df_count)):
    print("PASS")
else:
  print("FAIL")


In [0]:
good_df.write.mode("overwrite").format("delta").option('mergeSchema', 'true').saveAsTable('it_support_tickets.bronze.bronze_table')

In [0]:
bad_df.write.mode("overwrite").format("delta").option('mergeSchema', 'true').saveAsTable('it_support_tickets.bronze.quarantine_table')


In [0]:

rejection_reasons = spark.read.table("it_support_tickets.bronze.quarantine_table")



In [0]:
# adding rejection records to each rejected record
from pyspark.sql.functions import when, col
rejection_reasons = rejection_reasons.withColumn("rejection_reason", when(col("created_date").isNull(), "Invalid Date").otherwise(when(col("employee_id").isNull(), "Invalid Employee ID").otherwise(when(~col("employee_id").rlike("^E[0-9]{3}$"), "Invalid Employee ID").otherwise(when(col("support_category").isNull(), "Invalid Support Category").otherwise(when(col("priority").isNull(), "Invalid Priority").otherwise(when(col("support_engineer").isNull(), "Invalid Support Engineer").otherwise(when(~col("support_engineer").isin(valid_engineers), "Invalid Support Engineer").otherwise(when(col("location").isNull(), "Invalid Location").otherwise(when(~col("location").isin(valid_locations), "Invalid Location").otherwise(when(col("resolution_hours").isNull(), "Invalid Resolution Hours").otherwise(when(col("resolution_hours") <= 0, "Invalid Resolution Hours").otherwise(when(col("hourly_cost").isNull(), "Invalid Hourly Cost").otherwise(when(col("hourly_cost") <= 0, "Invalid Hourly Cost").otherwise(when(col("ticket_status").isNull(), "Invalid Ticket Status").otherwise(when(~col("ticket_status").isin(["Open", "Closed"]), "Invalid Ticket Status"))))))))))))))))

display(rejection_reasons)
# adding rejection records to each rejected record

In [0]:
rejection_reasons.write.mode("overwrite").format("delta").option('mergeSchema', 'true').saveAsTable('it_support_tickets.bronze.quarantine_table')
